In [ ]:
import pandas as pd
import numpy as np
import h5py
import json
import glob
import os
import sqlite3
import h5py
import matplotlib.pyplot as plt
from concurrent import futures
from itertools import repeat

In [ ]:
mf = "/projects/vsokolov/austin-transit-focus/"
expfolder="experiments"
output_dir_name = "Austin"
metrics_file = 'vals-metric-transit-focus.csv'

In [ ]:
# Check if all are complete
for folder in  glob.glob(f'{mf}/{expfolder}/**'):
    if not os.path.isfile(f'{folder}/{output_dir_name}/finished'):
        print(f'{folder}/finished')

In [ ]:
# EDA
def plotsum(f):
    sum = pd.read_csv(f'{f}/summary.csv', index_col=False)['in_network'].values
    with h5py.File(f'{f}/Austin-Result.h5', 'r') as f:
        # ref_output = f['link_moe']['link_travel_time'][:]*f['link_moe']['link_in_volume'][:]
        ref_output = f['link_moe']['num_vehicles_in_link'][:]
        ref_output =  np.sum(ref_output, axis=1)
    plt.close()
    fig, axs = plt.subplots(1,2,figsize=(6,2))
    axs[0].plot(sum)
    axs[1].plot(ref_output)

In [ ]:
plotsum('/home/vsokolov/Austin/Austin_iteration_10/')

In [ ]:
plotsum(f'../{expfolder}/Sim10/{output_dir_name}')

In [ ]:
with open(f"{mf}/config_morris_transit.json",'r') as fh:
    config = json.load(fh)
polaris_json_file = list(config.keys())[0]
vars = [item['name'] for item in config[polaris_json_file]]
print(polaris_json_file)
print(vars)

control_files = glob.glob(f'{mf}/{expfolder}/**/{polaris_json_file}')
with open(control_files[0]) as fh:
    data = json.load(fh)
root_key = list(data.keys())[0]
root_key

In [ ]:
d = []
for sim in control_files:    
    with open(sim) as fh:
        data = json.load(fh)
    vals = [data[root_key][item] for item in vars]
    vals= vals+[os.path.dirname(sim)]
    d.append(vals)

In [ ]:
df = pd.DataFrame(d, columns=vars+['folder'])
df.iloc[0:3]
df.tail()
# df.to_csv('vals-folder.csv', index=False)

In [ ]:
f = df.loc[0,'folder']
con = sqlite3.connect(f"{f}/{output_dir_name}/Austin-Demand.sqlite")
res = con.execute("select mode, count(*) from Trip group by mode").fetchall()
mode, share = ([i for i, j in res],[j for i, j in res])
print(mode,share)
for i in mode:
    df[f'm{i}'] = 0
df.head()
df.tail()

In [ ]:
def get_demand(i):
    f = df.loc[i,'folder']
    with open(f'{f}/{output_dir_name}/log/polaris_progress.log','r') as fh:
        if fh.readlines() [-3:][0].find('Finished') == -1:
            return(False, f"{f} is not finished, skipping")
    con = sqlite3.connect(f"{f}/{output_dir_name}/Austin-Demand.sqlite")
    res = con.execute("select mode, count(*) from Trip group by mode").fetchall()
    mode, share = ([i for i, j in res],[j for i, j in res])
    for j in range(len(mode)):
        df.loc[i,f'm{mode[j]}'] = share[j]#/sum(share)
    return(True, f"{f} {share[0]}")

In [ ]:
with futures.ThreadPoolExecutor(20) as executor:
    result = executor.map(get_demand, range(len(df)))
    for res in result:
        flag, msg = res
        if not flag:
            print(msg)

In [ ]:
# df.to_csv(metrics_file, index=False)
df = pd.read_csv(metrics_file)
df.head()
df.tail()

In [ ]:
(df['m0']/df['m0'].sum()).plot()

In [ ]:
# f = h5py.File("/home/vsokolov/Austin/Austin_iteration_10/Austin-Result.h5", 'r')
# f['link_moe'].keys()

In [ ]:
df['net'] = 0.0
df.head()

In [ ]:
def get_network_metrics(db):
    with h5py.File(db, 'r') as f:
        # ref_output = f['link_moe']['link_travel_time'][:]*f['link_moe']['link_in_volume'][:]
        res = f['link_moe']['num_vehicles_in_link'][:]
        res =  np.sum(res, axis=1)
    return res
def get_network(i,refout):
    f = df.loc[i,'folder']
    db = f'{f}/{output_dir_name}/Austin-Result.h5'
    out = get_network_metrics(db)
    res = np.mean(np.abs(out-refout))
    df.loc[i,'net'] = res
    # print(f"{f} {res}")
    return(res)

In [ ]:
refout = get_network_metrics("/home/vsokolov/Austin/Austin_iteration_10/Austin-Result.h5")
plt.plot(refout)

In [ ]:
get_network(0,refout)
df.head()

In [ ]:
with futures.ThreadPoolExecutor(10) as executor:
    result = executor.map(get_network, range(len(df)),repeat(refout))
    for res in result:
        print(res)  

In [ ]:
df.head()
df.to_csv(metrics_file, index=False)